# Side Effect Structural Audit

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 40.


In [ ]:
# ============================================================
# TASK 3 — FINAL SPLIT + ANNOTATION STRUCTURAL AUDIT
# ============================================================
#
# NO TRAINING
#
# Audits:
# 1) Saved SideEffect split sizes / class-target supports
# 2) Morgan-input overlap across Train / Cal / Test
# 3) Morgan-input + target overlap across splits
# 4) Interaction -> SideEffect-vector determinism
# 5) SideEffect-vector redundancy across Interaction classes
# 6) Train-only lookup structure
# 7) Reviewer-ready tables + wording
#
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp

warnings.filterwarnings("ignore")


# ============================================================
# 1) ROOTS
# ============================================================

SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

INT_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-v2-menna-projectt"
)

OUT = Path(
    "/kaggle/working/"
    "FINAL_SPLIT_ANNOTATION_AUDIT"
)

if OUT.exists():
    shutil.rmtree(OUT)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2) KNOWN FILES
# ============================================================

FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_CAL_PATH = (
    SIDE_ROOT /
    "X_calibration.npz"
)

X_TEST_PATH = (
    SIDE_ROOT /
    "X_test.npz"
)

Y_CAL_PATH = (
    SIDE_ROOT /
    "Y_calibration.npy"
)

Y_TEST_PATH = (
    SIDE_ROOT /
    "Y_test.npy"
)

INT_TRAIN_ROWS = (
    INT_ROOT /
    "train_rows.csv"
)


for p in [
    FULL_PATH,
    LABEL_MAP_PATH,
    X_CAL_PATH,
    X_TEST_PATH,
    Y_CAL_PATH,
    Y_TEST_PATH,
    INT_TRAIN_ROWS
]:
    assert p.exists(), p


# ============================================================
# 3) DISCOVER SIDE-EFFECT X_train + Y_train SAFELY BY SHAPE
# ============================================================

print("=" * 120)
print("DISCOVERING SIDE-EFFECT TRAIN ARRAYS")
print("=" * 120)


def discover_sparse_matrix(
    expected_shape,
    filename_token="train"
):

    candidates = []

    # Side dataset first
    for p in SIDE_ROOT.rglob("*.npz"):

        if filename_token.lower() not in p.name.lower():
            continue

        try:
            X = sp.load_npz(p)

            if X.shape == expected_shape:
                candidates.append(p)

        except Exception:
            pass


    # Broader search only if needed
    if not candidates:

        broad_root = Path(
            "/kaggle/input/datasets/"
            "amiramohammeddd"
        )

        for p in broad_root.rglob("*.npz"):

            s = str(p).lower()

            if (
                "cold" in s
                or
                "scaffold" in s
                or
                "mhafr" in s
                or
                "autoddi" in s
                or
                "coordinates" in s
            ):
                continue

            if filename_token.lower() not in p.name.lower():
                continue

            try:
                X = sp.load_npz(p)

                if X.shape == expected_shape:
                    candidates.append(p)

            except Exception:
                pass


    candidates = list(
        dict.fromkeys(candidates)
    )

    print(
        f"Candidates for {expected_shape}:"
    )

    for p in candidates:
        print(" •", p)


    if len(candidates) == 0:
        return None

    if len(candidates) == 1:
        return candidates[0]


    # Prefer files from SIDE_ROOT
    side_candidates = [
        p
        for p in candidates
        if str(p).startswith(
            str(SIDE_ROOT)
        )
    ]

    if len(side_candidates) == 1:
        return side_candidates[0]


    raise RuntimeError(
        f"Multiple matrices match {expected_shape}. "
        f"Inspect printed candidates before proceeding."
    )


def discover_y_train():

    candidates = []

    for p in SIDE_ROOT.rglob("*.npy"):

        if "train" not in p.name.lower():
            continue

        if "prob" in p.name.lower():
            continue

        try:
            arr = np.load(
                p,
                mmap_mode="r"
            )

            if arr.shape == (
                134249,
                263
            ):
                candidates.append(p)

        except Exception:
            pass


    print(
        "\nY_train candidates:"
    )

    for p in candidates:
        print(" •", p)


    if len(candidates) == 1:
        return candidates[0]

    preferred = [
        p
        for p in candidates
        if p.name == "Y_train (1).npy"
    ]

    if len(preferred) == 1:
        return preferred[0]

    if len(candidates) == 0:
        return None

    raise RuntimeError(
        "Multiple Y_train arrays with expected shape."
    )


X_TRAIN_PATH = discover_sparse_matrix(
    expected_shape=(
        134249,
        4096
    ),
    filename_token="train"
)

Y_TRAIN_PATH = discover_y_train()


print("\nSelected:")
print("X_train:", X_TRAIN_PATH)
print("Y_train:", Y_TRAIN_PATH)


assert Y_TRAIN_PATH is not None


# ============================================================
# 4) LOAD CORE DATA
# ============================================================

full_df = pd.read_csv(
    FULL_PATH
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)

train_rows = pd.read_csv(
    INT_TRAIN_ROWS
)

Y_train = np.load(
    Y_TRAIN_PATH
).astype(
    np.uint8
)

Y_cal = np.load(
    Y_CAL_PATH
).astype(
    np.uint8
)

Y_test = np.load(
    Y_TEST_PATH
).astype(
    np.uint8
)

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

if X_TRAIN_PATH is not None:

    X_train = sp.load_npz(
        X_TRAIN_PATH
    ).tocsr()

else:

    X_train = None


print("\n" + "=" * 120)
print("CORE DATA")
print("=" * 120)

print("Final dataset :", full_df.shape)
print("Label map     :", label_map.shape)
print("Y_train       :", Y_train.shape)
print("Y_cal         :", Y_cal.shape)
print("Y_test        :", Y_test.shape)

if X_train is not None:
    print("X_train       :", X_train.shape)

print("X_cal         :", X_cal.shape)
print("X_test        :", X_test.shape)


assert len(full_df) == 191795
assert label_map.shape[0] == 263

assert Y_train.shape == (
    134249,
    263
)

assert Y_cal.shape == (
    28768,
    263
)

assert Y_test.shape == (
    28768,
    263
)

assert X_cal.shape == (
    28768,
    4096
)

assert X_test.shape == (
    28768,
    4096
)

if X_train is not None:

    assert X_train.shape == (
        134249,
        4096
    )


# ============================================================
# 5) DATASET CLASS QA
# ============================================================

print("\n" + "=" * 120)
print("ANNOTATION CLASS QA")
print("=" * 120)


required_cols = [
    "Drug1_ID",
    "Drug1",
    "Drug2_ID",
    "Drug2",
    "Interaction",
    "Label",
    "sideeffect_str"
]

for c in required_cols:
    assert c in full_df.columns, c


print(
    "Interaction IDs:",
    full_df[
        "Interaction"
    ].nunique()
)

print(
    "Exact Interaction labels:",
    full_df[
        "Label"
    ].nunique()
)

print(
    "Computational SideEffect labels:",
    len(
        label_map
    )
)


assert full_df[
    "Interaction"
].nunique() == 84

assert full_df[
    "Label"
].nunique() == 84


# ============================================================
# 6) LABEL VOCABULARY
# ============================================================

side_label_to_id = dict(
    zip(
        label_map[
            "label"
        ].astype(str),
        label_map[
            "label_id"
        ].astype(int)
    )
)


assert len(
    side_label_to_id
) == 263


def target_signature(
    value
):

    terms = [
        x.strip()
        for x in str(
            value
        ).split("|")
        if x.strip()
    ]

    unknown = [
        x
        for x in terms
        if x not in side_label_to_id
    ]

    if unknown:

        raise KeyError(
            f"Unknown labels: {unknown}"
        )


    return tuple(
        sorted(
            side_label_to_id[x]
            for x in terms
        )
    )


full_df[
    "_Target_Signature"
] = (
    full_df[
        "sideeffect_str"
    ]
    .map(
        target_signature
    )
)


full_df[
    "_Target_Cardinality"
] = (
    full_df[
        "_Target_Signature"
    ]
    .map(len)
)


# ============================================================
# 7) INTERACTION → TARGET DETERMINISM
# ============================================================

class_determinism = (
    full_df
    .groupby(
        [
            "Interaction",
            "Label"
        ],
        as_index=False
    )
    .agg(
        Rows=(
            "sideeffect_str",
            "size"
        ),

        Unique_SideEffect_Strings=(
            "sideeffect_str",
            "nunique"
        ),

        Unique_Target_Vectors=(
            "_Target_Signature",
            "nunique"
        ),

        Target_Cardinality_Min=(
            "_Target_Cardinality",
            "min"
        ),

        Target_Cardinality_Max=(
            "_Target_Cardinality",
            "max"
        )
    )
)


print("\n" + "=" * 120)
print("INTERACTION → SIDEEFFECT VECTOR DETERMINISM")
print("=" * 120)

print(
    "Interaction classes:",
    len(
        class_determinism
    )
)

print(
    "Classes with exactly ONE target vector:",
    int(
        (
            class_determinism[
                "Unique_Target_Vectors"
            ]
            ==
            1
        ).sum()
    )
)

print(
    "Classes with >1 target vectors:",
    int(
        (
            class_determinism[
                "Unique_Target_Vectors"
            ]
            >
            1
        ).sum()
    )
)


DETERMINISTIC_ALL = bool(
    (
        class_determinism[
            "Unique_Target_Vectors"
        ]
        ==
        1
    ).all()
)


print(
    "\nDETERMINISTIC_ALL =",
    DETERMINISTIC_ALL
)


assert len(
    class_determinism
) == 84

assert DETERMINISTIC_ALL


# ============================================================
# 8) ONE CANONICAL VECTOR PER INTERACTION CLASS
# ============================================================

class_vectors = (
    full_df[
        [
            "Interaction",
            "Label",
            "_Target_Signature",
            "_Target_Cardinality"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "Interaction"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    class_vectors
) == 84


class_vectors[
    "Target_Vector_String"
] = (
    class_vectors[
        "_Target_Signature"
    ]
    .map(
        lambda x:
        "|".join(
            map(
                str,
                x
            )
        )
    )
)


# ============================================================
# 9) TARGET-VECTOR REDUNDANCY ACROSS INTERACTION CLASSES
# ============================================================

vector_groups = (
    class_vectors
    .groupby(
        "Target_Vector_String",
        as_index=False
    )
    .agg(
        Interaction_Class_Count=(
            "Interaction",
            "nunique"
        ),

        Interaction_IDs=(
            "Interaction",
            lambda x:
            " | ".join(
                map(
                    str,
                    sorted(
                        x.astype(int)
                        .unique()
                    )
                )
            )
        ),

        Interaction_Labels=(
            "Label",
            lambda x:
            " || ".join(
                sorted(
                    x.astype(str)
                    .unique()
                )
            )
        ),

        Target_Cardinality=(
            "_Target_Cardinality",
            "first"
        )
    )
)


unique_target_vectors = len(
    vector_groups
)

shared_vector_groups = int(
    (
        vector_groups[
            "Interaction_Class_Count"
        ]
        >
        1
    ).sum()
)

classes_in_shared_vectors = int(
    vector_groups.loc[
        vector_groups[
            "Interaction_Class_Count"
        ]
        >
        1,
        "Interaction_Class_Count"
    ].sum()
)

max_classes_same_vector = int(
    vector_groups[
        "Interaction_Class_Count"
    ].max()
)


print("\n" + "=" * 120)
print("TARGET-VECTOR REDUNDANCY")
print("=" * 120)

print(
    "Interaction classes:",
    84
)

print(
    "Unique 263-label target vectors:",
    unique_target_vectors
)

print(
    "Shared target-vector groups:",
    shared_vector_groups
)

print(
    "Interaction classes participating "
    "in shared target vectors:",
    classes_in_shared_vectors
)

print(
    "Maximum Interaction classes "
    "sharing one vector:",
    max_classes_same_vector
)


# ============================================================
# 10) CONDITIONAL ENTROPY STATEMENT
# ============================================================

# Because every Interaction class maps to exactly one target vector:
# H(TargetVector | InteractionClass) = 0 empirically.

CONDITIONAL_ENTROPY_ZERO = (
    DETERMINISTIC_ALL
)


# ============================================================
# 11) TRAIN-ONLY LOOKUP AUDIT
# ============================================================

print("\n" + "=" * 120)
print("TRAIN-ONLY LOOKUP STRUCTURE")
print("=" * 120)


assert "Label" in train_rows.columns
assert "sideeffect_str" in train_rows.columns


train_rows[
    "_Target_Signature"
] = (
    train_rows[
        "sideeffect_str"
    ]
    .map(
        target_signature
    )
)


train_lookup_audit = (
    train_rows
    .groupby(
        "Label",
        as_index=False
    )
    .agg(
        Train_Rows=(
            "_Target_Signature",
            "size"
        ),

        Unique_Target_Vectors=(
            "_Target_Signature",
            "nunique"
        )
    )
)


print(
    "Interaction classes represented:",
    train_lookup_audit[
        "Label"
    ].nunique()
)

print(
    "Classes with one train target vector:",
    int(
        (
            train_lookup_audit[
                "Unique_Target_Vectors"
            ]
            ==
            1
        ).sum()
    )
)

print(
    "Classes with >1 train target vector:",
    int(
        (
            train_lookup_audit[
                "Unique_Target_Vectors"
            ]
            >
            1
        ).sum()
    )
)


assert train_lookup_audit[
    "Label"
].nunique() == 84

assert (
    train_lookup_audit[
        "Unique_Target_Vectors"
    ]
    ==
    1
).all()


# ============================================================
# 12) Y SUPPORT / PREVALENCE SPLIT AUDIT
# ============================================================

split_target_summary = pd.DataFrame(
    {
        "Split": [
            "Train",
            "Calibration",
            "Test"
        ],

        "Rows": [
            Y_train.shape[0],
            Y_cal.shape[0],
            Y_test.shape[0]
        ],

        "Total_Positive_Labels": [
            int(
                Y_train.sum()
            ),
            int(
                Y_cal.sum()
            ),
            int(
                Y_test.sum()
            )
        ],

        "Mean_Labels_Per_Row": [
            float(
                Y_train.sum(
                    axis=1
                ).mean()
            ),
            float(
                Y_cal.sum(
                    axis=1
                ).mean()
            ),
            float(
                Y_test.sum(
                    axis=1
                ).mean()
            )
        ],

        "Median_Labels_Per_Row": [
            float(
                np.median(
                    Y_train.sum(
                        axis=1
                    )
                )
            ),
            float(
                np.median(
                    Y_cal.sum(
                        axis=1
                    )
                )
            ),
            float(
                np.median(
                    Y_test.sum(
                        axis=1
                    )
                )
            )
        ],

        "Positive_Cell_Prevalence": [
            float(
                Y_train.mean()
            ),
            float(
                Y_cal.mean()
            ),
            float(
                Y_test.mean()
            )
        ]
    }
)


print("\n" + "=" * 120)
print("TARGET DISTRIBUTION ACROSS SIDE-EFFECT SPLITS")
print("=" * 120)

print(
    split_target_summary.to_string(
        index=False
    )
)


# ============================================================
# 13) SPARSE ROW HASH HELPERS
# ============================================================

def sparse_row_hashes(
    X
):

    X = X.tocsr(
        copy=True
    )

    X.sort_indices()


    integer_like = np.allclose(
        X.data,
        np.round(
            X.data
        )
    )


    hashes = []


    for i in range(
        X.shape[0]
    ):

        a = X.indptr[i]
        b = X.indptr[i + 1]


        idx = X.indices[
            a:b
        ].astype(
            np.int32,
            copy=False
        )


        vals = X.data[
            a:b
        ]


        if integer_like:

            vals = np.round(
                vals
            ).astype(
                np.int16
            )

        else:

            vals = np.round(
                vals.astype(
                    np.float64
                ),
                7
            ).astype(
                np.float32
            )


        h = hashlib.sha1()

        h.update(
            idx.tobytes()
        )

        h.update(
            vals.tobytes()
        )

        hashes.append(
            h.hexdigest()
        )


    return hashes


def y_row_hashes(
    Y
):

    hashes = []

    for i in range(
        Y.shape[0]
    ):

        active = np.flatnonzero(
            Y[i]
        ).astype(
            np.int16
        )

        h = hashlib.sha1(
            active.tobytes()
        ).hexdigest()

        hashes.append(
            h
        )

    return hashes


# ============================================================
# 14) MODEL-INPUT OVERLAP
# ============================================================

representation_overlap_rows = []


if X_train is not None:

    print("\n" + "=" * 120)
    print("HASHING SAVED SIDE-EFFECT MODEL INPUTS")
    print("=" * 120)


    H_train = sparse_row_hashes(
        X_train
    )

    H_cal = sparse_row_hashes(
        X_cal
    )

    H_test = sparse_row_hashes(
        X_test
    )


    YH_train = y_row_hashes(
        Y_train
    )

    YH_cal = y_row_hashes(
        Y_cal
    )

    YH_test = y_row_hashes(
        Y_test
    )


    C_train = list(
        zip(
            H_train,
            YH_train
        )
    )

    C_cal = list(
        zip(
            H_cal,
            YH_cal
        )
    )

    C_test = list(
        zip(
            H_test,
            YH_test
        )
    )


    split_hashes = {
        "Train": H_train,
        "Calibration": H_cal,
        "Test": H_test
    }


    split_composite = {
        "Train": C_train,
        "Calibration": C_cal,
        "Test": C_test
    }


    pairs = [
        (
            "Train",
            "Calibration"
        ),
        (
            "Train",
            "Test"
        ),
        (
            "Calibration",
            "Test"
        )
    ]


    for A, B in pairs:

        A_hash = split_hashes[
            A
        ]

        B_hash = split_hashes[
            B
        ]


        A_set = set(
            A_hash
        )

        B_set = set(
            B_hash
        )


        B_seen_in_A = np.asarray(
            [
                x in A_set
                for x in B_hash
            ]
        )


        unique_overlap = (
            A_set
            &
            B_set
        )


        A_comp = set(
            split_composite[
                A
            ]
        )

        B_comp_list = (
            split_composite[
                B
            ]
        )


        B_same_input_target = np.asarray(
            [
                x in A_comp
                for x in B_comp_list
            ]
        )


        representation_overlap_rows.append(
            {
                "Reference_Split":
                    A,

                "Evaluated_Split":
                    B,

                "Reference_Rows":
                    len(
                        A_hash
                    ),

                "Evaluated_Rows":
                    len(
                        B_hash
                    ),

                "Unique_Input_Hashes_Reference":
                    len(
                        A_set
                    ),

                "Unique_Input_Hashes_Evaluated":
                    len(
                        B_set
                    ),

                "Unique_Input_Hash_Intersection":
                    len(
                        unique_overlap
                    ),

                "Evaluated_Rows_Input_Seen_In_Reference":
                    int(
                        B_seen_in_A.sum()
                    ),

                "Evaluated_Rows_Input_Seen_Rate":
                    float(
                        B_seen_in_A.mean()
                    ),

                "Evaluated_Rows_Same_Input_And_Target_Seen":
                    int(
                        B_same_input_target.sum()
                    ),

                "Evaluated_Rows_Same_Input_And_Target_Seen_Rate":
                    float(
                        B_same_input_target.mean()
                    )
            }
        )


    representation_overlap = pd.DataFrame(
        representation_overlap_rows
    )


    print("\n" + "=" * 120)
    print("SAVED MORGAN INPUT OVERLAP")
    print("=" * 120)

    print(
        representation_overlap.to_string(
            index=False
        )
    )


else:

    representation_overlap = pd.DataFrame(
        [
            {
                "Note":
                    "X_train with shape (134249,4096) "
                    "was not found; representation-overlap "
                    "audit could not be computed."
            }
        ]
    )


# ============================================================
# 15) DUPLICATION WITHIN EACH SAVED SPLIT
# ============================================================

within_split_rows = []


if X_train is not None:

    for split_name, hashes in [
        (
            "Train",
            H_train
        ),
        (
            "Calibration",
            H_cal
        ),
        (
            "Test",
            H_test
        )
    ]:

        counts = Counter(
            hashes
        )

        duplicate_rows = sum(
            count
            for count in counts.values()
            if count > 1
        )


        within_split_rows.append(
            {
                "Split":
                    split_name,

                "Rows":
                    len(
                        hashes
                    ),

                "Unique_Model_Inputs":
                    len(
                        counts
                    ),

                "Rows_Belonging_to_Repeated_Input":
                    int(
                        duplicate_rows
                    ),

                "Repeated_Input_Row_Rate":
                    float(
                        duplicate_rows
                        /
                        len(
                            hashes
                        )
                    )
            }
        )


within_split = pd.DataFrame(
    within_split_rows
)


# ============================================================
# 16) SIDE LABEL SUPPORT STABILITY
# ============================================================

support_table = pd.DataFrame(
    {
        "label_id":
            np.arange(
                263
            ),

        "label":
            label_map[
                "label"
            ].astype(str)
            .values,

        "Train_Support":
            Y_train.sum(
                axis=0
            ).astype(int),

        "Calibration_Support":
            Y_cal.sum(
                axis=0
            ).astype(int),

        "Test_Support":
            Y_test.sum(
                axis=0
            ).astype(int)
    }
)


support_table[
    "Train_Prevalence"
] = (
    support_table[
        "Train_Support"
    ]
    /
    len(
        Y_train
    )
)

support_table[
    "Calibration_Prevalence"
] = (
    support_table[
        "Calibration_Support"
    ]
    /
    len(
        Y_cal
    )
)

support_table[
    "Test_Prevalence"
] = (
    support_table[
        "Test_Support"
    ]
    /
    len(
        Y_test
    )
)


# ============================================================
# 17) SUMMARY
# ============================================================

summary_rows = [
    {
        "Item":
            "Final dataset rows",

        "Value":
            len(
                full_df
            )
    },

    {
        "Item":
            "Interaction classes",

        "Value":
            full_df[
                "Interaction"
            ].nunique()
    },

    {
        "Item":
            "Computational side-effect labels",

        "Value":
            len(
                label_map
            )
    },

    {
        "Item":
            "Interaction classes with exactly one target vector",

        "Value":
            int(
                (
                    class_determinism[
                        "Unique_Target_Vectors"
                    ]
                    ==
                    1
                ).sum()
            )
    },

    {
        "Item":
            "Interaction classes with >1 target vector",

        "Value":
            int(
                (
                    class_determinism[
                        "Unique_Target_Vectors"
                    ]
                    >
                    1
                ).sum()
            )
    },

    {
        "Item":
            "Empirical H(TargetVector | Interaction)",

        "Value":
            0.0
    },

    {
        "Item":
            "Unique target vectors among 84 Interaction classes",

        "Value":
            unique_target_vectors
    },

    {
        "Item":
            "Shared target-vector groups",

        "Value":
            shared_vector_groups
    },

    {
        "Item":
            "Classes involved in shared target vectors",

        "Value":
            classes_in_shared_vectors
    },

    {
        "Item":
            "Max classes sharing one target vector",

        "Value":
            max_classes_same_vector
    },

    {
        "Item":
            "Train-only lookup Interaction classes",

        "Value":
            train_lookup_audit[
                "Label"
            ].nunique()
    },

    {
        "Item":
            "Train-only lookup classes with one target vector",

        "Value":
            int(
                (
                    train_lookup_audit[
                        "Unique_Target_Vectors"
                    ]
                    ==
                    1
                ).sum()
            )
    }
]


summary = pd.DataFrame(
    summary_rows
)


print("\n" + "=" * 120)
print("FINAL STRUCTURAL AUDIT SUMMARY")
print("=" * 120)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 18) REVIEWER-READY TEXT
# ============================================================

if X_train is not None:

    train_test_overlap_row = (
        representation_overlap[
            (
                representation_overlap[
                    "Reference_Split"
                ]
                ==
                "Train"
            )
            &
            (
                representation_overlap[
                    "Evaluated_Split"
                ]
                ==
                "Test"
            )
        ]
        .iloc[0]
    )


    train_test_input_seen = float(
        train_test_overlap_row[
            "Evaluated_Rows_Input_Seen_Rate"
        ]
    )


    train_test_composite_seen = float(
        train_test_overlap_row[
            "Evaluated_Rows_Same_Input_And_Target_Seen_Rate"
        ]
    )


    overlap_sentence = (
        f"At the saved Morgan-input level, "
        f"{100*train_test_input_seen:.2f}% of test rows "
        f"had a 4096-bit ordered-pair representation "
        f"also present in the training split; "
        f"{100*train_test_composite_seen:.2f}% had both "
        f"the same model input and the same multilabel target "
        f"observed in training."
    )

else:

    overlap_sentence = (
        "The train-side Morgan matrix was not available "
        "in the attached artifact set, so cross-split "
        "model-input overlap was not quantified in this run."
    )


reviewer_text = f"""
Split and annotation audit

The side-effect task contains 134,249 training samples,
28,768 validation/calibration samples, and 28,768 test samples,
with 263 computational target labels.

A structural annotation audit across the final 84 interaction
classes showed that all {len(class_determinism)} interaction
classes mapped to exactly one multilabel side-effect target
vector. Thus, the empirical conditional entropy
H(TargetVector | InteractionClass) is zero under the current
annotation scheme.

Across the 84 interaction classes, there were
{unique_target_vectors} distinct 263-label target vectors.
{shared_vector_groups} target-vector patterns were shared by
more than one interaction class, involving
{classes_in_shared_vectors} interaction classes; the largest
shared pattern was associated with {max_classes_same_vector}
interaction classes.

The train-only audit independently confirmed that all
{train_lookup_audit['Label'].nunique()} interaction classes
represented in the interaction training set were associated
with exactly one side-effect target vector.

{overlap_sentence}

These findings indicate strong class-level structural
dependence between the interaction-type annotation and the
side-effect target. Therefore, side-effect performance under
the current row-random, interaction-stratified benchmark
should not be interpreted as fully independent molecular-level
generalization. The manuscript accordingly reports the
interaction-to-side-effect lookup as a structural baseline and
treats the independent LightGBM side-effect predictor as a
separate model evaluated under the existing annotation scheme.
""".strip()


print("\n" + "=" * 120)
print("REVIEWER-READY AUDIT TEXT")
print("=" * 120)

print(
    reviewer_text
)


# ============================================================
# 19) QA
# ============================================================

qa = pd.DataFrame(
    {
        "QA_Check": [
            "Final dataset has 84 Interaction classes",
            "Final vocabulary has 263 side-effect labels",
            "Every Interaction class has one target vector",
            "Empirical conditional entropy is zero",
            "Train-only lookup covers all 84 classes",
            "Every train-only lookup class has one target vector",
            "Saved Train/Cal/Test Y shapes correct",
            "Saved Cal/Test X shapes correct",
            "No training performed"
        ],

        "Passed": [
            full_df[
                "Interaction"
            ].nunique()
            ==
            84,

            len(
                label_map
            )
            ==
            263,

            DETERMINISTIC_ALL,

            CONDITIONAL_ENTROPY_ZERO,

            train_lookup_audit[
                "Label"
            ].nunique()
            ==
            84,

            bool(
                (
                    train_lookup_audit[
                        "Unique_Target_Vectors"
                    ]
                    ==
                    1
                ).all()
            ),

            (
                Y_train.shape
                ==
                (
                    134249,
                    263
                )
                and
                Y_cal.shape
                ==
                (
                    28768,
                    263
                )
                and
                Y_test.shape
                ==
                (
                    28768,
                    263
                )
            ),

            (
                X_cal.shape
                ==
                (
                    28768,
                    4096
                )
                and
                X_test.shape
                ==
                (
                    28768,
                    4096
                )
            ),

            True
        ]
    }
)


print("\n" + "=" * 120)
print("QA")
print("=" * 120)

print(
    qa.to_string(
        index=False
    )
)

assert qa[
    "Passed"
].all()


# ============================================================
# 20) SAVE CSV
# ============================================================

summary.to_csv(
    OUT /
    "01_Structural_Audit_Summary.csv",
    index=False
)

class_determinism.to_csv(
    OUT /
    "02_Interaction_to_Target_Determinism.csv",
    index=False
)

class_vectors.drop(
    columns=[
        "_Target_Signature"
    ]
).to_csv(
    OUT /
    "03_Interaction_Target_Vectors.csv",
    index=False
)

vector_groups.to_csv(
    OUT /
    "04_Shared_Target_Vector_Groups.csv",
    index=False
)

train_lookup_audit.to_csv(
    OUT /
    "05_Train_Only_Lookup_Audit.csv",
    index=False
)

split_target_summary.to_csv(
    OUT /
    "06_Split_Target_Distribution.csv",
    index=False
)

representation_overlap.to_csv(
    OUT /
    "07_Morgan_Input_Overlap.csv",
    index=False
)

within_split.to_csv(
    OUT /
    "08_Within_Split_Input_Duplication.csv",
    index=False
)

support_table.to_csv(
    OUT /
    "09_Label_Support_By_Split.csv",
    index=False
)

qa.to_csv(
    OUT /
    "10_QA.csv",
    index=False
)


# ============================================================
# 21) EXCEL
# ============================================================

XLSX_PATH = (
    OUT /
    "FINAL_SPLIT_ANNOTATION_AUDIT.xlsx"
)


with pd.ExcelWriter(
    XLSX_PATH,
    engine="openpyxl"
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    class_determinism.to_excel(
        writer,
        sheet_name="Class_Determinism",
        index=False
    )

    class_vectors.drop(
        columns=[
            "_Target_Signature"
        ]
    ).to_excel(
        writer,
        sheet_name="Class_Targets",
        index=False
    )

    vector_groups.to_excel(
        writer,
        sheet_name="Vector_Redundancy",
        index=False
    )

    train_lookup_audit.to_excel(
        writer,
        sheet_name="Train_Lookup",
        index=False
    )

    split_target_summary.to_excel(
        writer,
        sheet_name="Split_Distribution",
        index=False
    )

    representation_overlap.to_excel(
        writer,
        sheet_name="Input_Overlap",
        index=False
    )

    within_split.to_excel(
        writer,
        sheet_name="Within_Split_Duplicates",
        index=False
    )

    support_table.to_excel(
        writer,
        sheet_name="Label_Support",
        index=False
    )

    qa.to_excel(
        writer,
        sheet_name="QA",
        index=False
    )


# ============================================================
# 22) REVIEWER TEXT FILE
# ============================================================

TXT_PATH = (
    OUT /
    "REVIEWER_READY_SPLIT_ANNOTATION_AUDIT.txt"
)


with open(
    TXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        reviewer_text
    )


# ============================================================
# 23) JSON SUMMARY
# ============================================================

summary_json = {
    "final_rows":
        int(
            len(
                full_df
            )
        ),

    "interaction_classes":
        84,

    "sideeffect_labels":
        263,

    "train_rows":
        134249,

    "calibration_rows":
        28768,

    "test_rows":
        28768,

    "all_interaction_classes_deterministic":
        bool(
            DETERMINISTIC_ALL
        ),

    "conditional_entropy_target_given_interaction":
        0.0,

    "unique_target_vectors_among_84_classes":
        int(
            unique_target_vectors
        ),

    "shared_target_vector_groups":
        int(
            shared_vector_groups
        ),

    "classes_in_shared_vectors":
        int(
            classes_in_shared_vectors
        ),

    "max_classes_sharing_one_vector":
        int(
            max_classes_same_vector
        ),

    "train_lookup_classes":
        int(
            train_lookup_audit[
                "Label"
            ].nunique()
        ),

    "train_lookup_all_deterministic":
        bool(
            (
                train_lookup_audit[
                    "Unique_Target_Vectors"
                ]
                ==
                1
            ).all()
        ),

    "x_train_found":
        bool(
            X_train is not None
        ),

    "training_performed":
        False
}


if X_train is not None:

    summary_json[
        "test_rows_input_seen_in_train_rate"
    ] = float(
        train_test_input_seen
    )

    summary_json[
        "test_rows_same_input_target_seen_in_train_rate"
    ] = float(
        train_test_composite_seen
    )


with open(
    OUT /
    "FINAL_SPLIT_ANNOTATION_AUDIT.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary_json,
        f,
        indent=2
    )


# ============================================================
# 24) ZIP
# ============================================================

ZIP_PATH = shutil.make_archive(
    "/kaggle/working/"
    "FINAL_SPLIT_ANNOTATION_AUDIT",
    "zip",
    root_dir=OUT
)


# ============================================================
# 25) FINAL PRINT
# ============================================================

print("\n" + "=" * 120)
print("TASK 3 COMPLETE")
print("=" * 120)

print(
    "84/84 Interaction classes deterministic:",
    DETERMINISTIC_ALL
)

print(
    "Unique target vectors:",
    unique_target_vectors
)

print(
    "Shared-vector groups:",
    shared_vector_groups
)

print(
    "Conditional entropy "
    "H(TargetVector | Interaction): 0"
)


if X_train is not None:

    print(
        f"Test input seen in train: "
        f"{100*train_test_input_seen:.4f}%"
    )

    print(
        f"Test same input+target seen in train: "
        f"{100*train_test_composite_seen:.4f}%"
    )


print("\nExcel:")
print(XLSX_PATH)

print("\nReviewer text:")
print(TXT_PATH)

print("\nZIP:")
print(ZIP_PATH)

print("\n NO TRAINING PERFORMED")

Source cell: 41.


In [ ]:
# ============================================================
# TASK 3B — EXACT SIDE-EFFECT TRAIN INPUT OVERLAP
# via multiset subtraction
# ============================================================
#
# NO X_train FILE USED
# NO TRAINING
#
# Exact identity:
# Full molecular-valid universe
#   - saved calibration multiset
#   - saved test multiset
# = SideEffect training multiset
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json

import numpy as np
import pandas as pd
import scipy.sparse as sp

from rdkit import Chem
from rdkit.Chem import AllChem


SIDE_ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/"
    "ddi-unique263-cal-test-stage2"
)

FULL_PATH = (
    SIDE_ROOT /
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

LABEL_MAP_PATH = (
    SIDE_ROOT /
    "label_mapping.csv"
)

X_CAL_PATH = SIDE_ROOT / "X_calibration.npz"
X_TEST_PATH = SIDE_ROOT / "X_test.npz"

Y_CAL_PATH = SIDE_ROOT / "Y_calibration.npy"
Y_TEST_PATH = SIDE_ROOT / "Y_test.npy"

OUT = Path(
    "/kaggle/working/"
    "FINAL_SPLIT_ANNOTATION_AUDIT"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1) LOAD
# ============================================================

full_df = pd.read_csv(
    FULL_PATH
)

label_map = (
    pd.read_csv(
        LABEL_MAP_PATH
    )
    .sort_values("label_id")
    .reset_index(drop=True)
)

X_cal = sp.load_npz(
    X_CAL_PATH
).tocsr()

X_test = sp.load_npz(
    X_TEST_PATH
).tocsr()

Y_cal = np.load(
    Y_CAL_PATH
).astype(np.uint8)

Y_test = np.load(
    Y_TEST_PATH
).astype(np.uint8)


assert len(full_df) == 191795
assert X_cal.shape == (28768, 4096)
assert X_test.shape == (28768, 4096)
assert Y_cal.shape == (28768, 263)
assert Y_test.shape == (28768, 263)


# ============================================================
# 2) REMOVE THE SAME 10 MOLECULAR-INVALID ROWS
# ============================================================

all_smiles = pd.unique(
    pd.concat(
        [
            full_df["Drug1"],
            full_df["Drug2"]
        ],
        ignore_index=True
    )
)

validity = {}

for x in all_smiles:

    if pd.isna(x):
        validity[x] = False
    else:
        validity[x] = (
            Chem.MolFromSmiles(
                str(x)
            )
            is not None
        )


valid_mask = (
    full_df["Drug1"].map(
        lambda x:
        False
        if pd.isna(x)
        else validity[x]
    )
    &
    full_df["Drug2"].map(
        lambda x:
        False
        if pd.isna(x)
        else validity[x]
    )
)


valid_df = (
    full_df.loc[
        valid_mask
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 110)
print("VALID UNIVERSE")
print("=" * 110)

print("Rows:", len(valid_df))

assert len(valid_df) == 191785


# ============================================================
# 3) LABEL TARGET SIGNATURE
# ============================================================

label_to_id = dict(
    zip(
        label_map["label"].astype(str),
        label_map["label_id"].astype(int)
    )
)


def target_signature(value):

    terms = [
        x.strip()
        for x in str(value).split("|")
        if x.strip()
    ]

    return tuple(
        sorted(
            label_to_id[x]
            for x in terms
        )
    )


full_target_sig = (
    valid_df["sideeffect_str"]
    .map(target_signature)
    .tolist()
)


def y_signatures(Y):

    return [
        tuple(
            np.flatnonzero(
                Y[i]
            )
            .astype(int)
            .tolist()
        )
        for i in range(
            Y.shape[0]
        )
    ]


cal_y_sig = y_signatures(
    Y_cal
)

test_y_sig = y_signatures(
    Y_test
)


# ============================================================
# 4) BUILD EXACT MORGAN BIT HASH FOR FULL UNIVERSE
# ============================================================

unique_smiles = sorted(
    set(
        valid_df["Drug1"].astype(str)
    )
    |
    set(
        valid_df["Drug2"].astype(str)
    )
)


fp_bits = {}


print("\nBuilding Morgan cache...")


for n, smi in enumerate(
    unique_smiles,
    start=1
):

    mol = Chem.MolFromSmiles(
        smi
    )

    assert mol is not None

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048,
        useChirality=False,
        useFeatures=False
    )

    fp_bits[smi] = np.asarray(
        fp.GetOnBits(),
        dtype=np.int32
    )

    if n % 500 == 0:
        print(
            f"{n:,}/{len(unique_smiles):,}"
        )


def hash_pair(d1, d2):

    idx = np.concatenate(
        [
            fp_bits[str(d1)],
            fp_bits[str(d2)] + 2048
        ]
    ).astype(np.int32)

    vals = np.ones(
        len(idx),
        dtype=np.int16
    )

    h = hashlib.sha1()

    h.update(
        idx.tobytes()
    )

    h.update(
        vals.tobytes()
    )

    return h.hexdigest()


print("\nHashing full universe...")


full_input_hash = []

for i, row in enumerate(
    valid_df.itertuples(index=False)
):

    full_input_hash.append(
        hash_pair(
            row.Drug1,
            row.Drug2
        )
    )

    if (i + 1) % 25000 == 0:
        print(
            f"{i+1:,}/{len(valid_df):,}"
        )


# ============================================================
# 5) HASH SAVED CAL / TEST X
# ============================================================

def csr_hashes(X):

    X = X.tocsr(copy=True)
    X.sort_indices()

    assert np.allclose(
        X.data,
        np.round(X.data)
    )

    out = []

    for i in range(
        X.shape[0]
    ):

        a = X.indptr[i]
        b = X.indptr[i + 1]

        idx = X.indices[
            a:b
        ].astype(
            np.int32,
            copy=False
        )

        vals = np.round(
            X.data[a:b]
        ).astype(
            np.int16
        )

        h = hashlib.sha1()

        h.update(
            idx.tobytes()
        )

        h.update(
            vals.tobytes()
        )

        out.append(
            h.hexdigest()
        )

    return out


cal_input_hash = csr_hashes(
    X_cal
)

test_input_hash = csr_hashes(
    X_test
)


# ============================================================
# 6) MULTISET SUBTRACTION
# ============================================================

FULL_INPUT = Counter(
    full_input_hash
)

CAL_INPUT = Counter(
    cal_input_hash
)

TEST_INPUT = Counter(
    test_input_hash
)


TRAIN_INPUT = (
    FULL_INPUT
    -
    CAL_INPUT
    -
    TEST_INPUT
)


print("\n" + "=" * 110)
print("INPUT MULTISET RECONSTRUCTION")
print("=" * 110)

print(
    "Full rows represented :",
    sum(
        FULL_INPUT.values()
    )
)

print(
    "Cal rows represented  :",
    sum(
        CAL_INPUT.values()
    )
)

print(
    "Test rows represented :",
    sum(
        TEST_INPUT.values()
    )
)

print(
    "Recovered train rows  :",
    sum(
        TRAIN_INPUT.values()
    )
)


assert sum(
    TRAIN_INPUT.values()
) == 134249


# ============================================================
# 7) TEST INPUT SEEN IN TRAIN
# ============================================================

train_input_keys = set(
    TRAIN_INPUT.keys()
)


test_seen_input = np.asarray(
    [
        h in train_input_keys
        for h in test_input_hash
    ]
)


print("\n" + "=" * 110)
print("EXACT SIDE-EFFECT MODEL-INPUT OVERLAP")
print("=" * 110)

print(
    "Test rows with Morgan input "
    "also present in SideEffect train:"
)

print(
    int(
        test_seen_input.sum()
    ),
    "/",
    len(
        test_seen_input
    )
)

print(
    "Rate:",
    f"{100*test_seen_input.mean():.4f}%"
)


# ============================================================
# 8) INPUT + TARGET MULTISET
# ============================================================

FULL_COMP = Counter(
    zip(
        full_input_hash,
        full_target_sig
    )
)

CAL_COMP = Counter(
    zip(
        cal_input_hash,
        cal_y_sig
    )
)

TEST_COMP = Counter(
    zip(
        test_input_hash,
        test_y_sig
    )
)


TRAIN_COMP = (
    FULL_COMP
    -
    CAL_COMP
    -
    TEST_COMP
)


assert sum(
    TRAIN_COMP.values()
) == 134249


train_comp_keys = set(
    TRAIN_COMP.keys()
)


test_seen_same_input_target = np.asarray(
    [
        key in train_comp_keys
        for key in zip(
            test_input_hash,
            test_y_sig
        )
    ]
)


print("\nTest rows with SAME Morgan input "
      "+ SAME 263-label target in train:")

print(
    int(
        test_seen_same_input_target.sum()
    ),
    "/",
    len(
        test_seen_same_input_target
    )
)

print(
    "Rate:",
    f"{100*test_seen_same_input_target.mean():.4f}%"
)


# ============================================================
# 9) CALIBRATION OVERLAP TOO
# ============================================================

cal_seen_input = np.asarray(
    [
        h in train_input_keys
        for h in cal_input_hash
    ]
)


cal_seen_same_input_target = np.asarray(
    [
        key in train_comp_keys
        for key in zip(
            cal_input_hash,
            cal_y_sig
        )
    ]
)


audit = pd.DataFrame(
    {
        "Evaluated_Split": [
            "Calibration",
            "Test"
        ],

        "Rows": [
            len(
                cal_input_hash
            ),
            len(
                test_input_hash
            )
        ],

        "Rows_Input_Seen_In_Train": [
            int(
                cal_seen_input.sum()
            ),
            int(
                test_seen_input.sum()
            )
        ],

        "Input_Seen_Rate": [
            float(
                cal_seen_input.mean()
            ),
            float(
                test_seen_input.mean()
            )
        ],

        "Rows_Same_Input_Target_Seen_In_Train": [
            int(
                cal_seen_same_input_target.sum()
            ),
            int(
                test_seen_same_input_target.sum()
            )
        ],

        "Same_Input_Target_Seen_Rate": [
            float(
                cal_seen_same_input_target.mean()
            ),
            float(
                test_seen_same_input_target.mean()
            )
        ]
    }
)


print("\n" + "=" * 110)
print("FINAL EXACT SIDE-EFFECT SPLIT OVERLAP")
print("=" * 110)

print(
    audit.to_string(
        index=False
    )
)


# ============================================================
# 10) SAVE
# ============================================================

audit.to_csv(
    OUT /
    "07B_EXACT_SIDE_EFFECT_INPUT_OVERLAP.csv",
    index=False
)


summary = {

    "method":
        "Full valid universe minus saved calibration and test multisets",

    "sideeffect_train_rows_recovered":
        int(
            sum(
                TRAIN_INPUT.values()
            )
        ),

    "test_rows":
        int(
            len(
                test_input_hash
            )
        ),

    "test_rows_input_seen_in_sideeffect_train":
        int(
            test_seen_input.sum()
        ),

    "test_input_seen_rate":
        float(
            test_seen_input.mean()
        ),

    "test_rows_same_input_target_seen_in_sideeffect_train":
        int(
            test_seen_same_input_target.sum()
        ),

    "test_same_input_target_seen_rate":
        float(
            test_seen_same_input_target.mean()
        ),

    "x_train_file_used":
        False,

    "training_performed":
        False
}


with open(
    OUT /
    "07B_EXACT_SIDE_EFFECT_INPUT_OVERLAP.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("\n" + "=" * 110)
print("FINAL SUMMARY")
print("=" * 110)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print(
    "\n No X_train file was assumed"
)

print(
    " Train multiset reconstructed exactly"
)

print(
    " Morgan collisions preserved by multiplicity"
)

print(
    " NO TRAINING PERFORMED"
)